# Introduction to images
::: {.content-visible when-format="pdf"}
We have been talking a lot about images; how we want to use them ,how they are formed etc. But we haven't defined what an image is and basic operations.
:::

In [ ]:
#| include: false
#| echo: false
# Setup cell: makes the notebook work in Jupyter *and* in both Quarto builds.
import os, sys
from pathlib import Path

if "COURSE_LECTURE_DIR" not in os.environ:          # interactive use
    os.environ["COURSE_LECTURE_DIR"] = str(Path.cwd())
    sys.path.insert(0, str(Path.cwd().parents[1] / "scripts"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from courselib import data_path, figure_path

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False})

## What is an image?

A very abstract definition: 

- A pairing between spatial information (position)
- and some other kind of information (value).

In most cases this is a 2- or 3-dimensional position with 

- x,y,z coordinates on a grid
- and a numeric value (intensity) or a vector (color).

***
### Image sampling

:::: {.columns}
::: {.column width=40% style="align-self: center;"}
| The world is | The computer needs|
|:---:|:---:|
| Continuous    | Discrete levels |
| No boundaries | Limited extent | 

:::

::: {.column width=60%}

![The real world is sampled into discrete images with limited extent.](figures/grid.png){#fig-imagesampling width=75%}

:::
::::

::: {.content-visible when-format="pdf"}

### What does sampling mean in practice

Sampling is the process of converting a signal into a discrete representation. Here, we see examples of both spatial and intensity sampling. The spatial sampling is decided by the pixel pitch of the detector while the intensity sampling tells how many levels we represent the intensity or amplitude of the measured signal. Sampling can also rely to time, in this case we often talk about frame rates. 

In [ ]:
#| echo: false
#| output: true
#| fig-cap: "Examples of image sampling in intensity and spatial domains."
#| label: fig-samplingwoodexample
from skimage.io import imread
from skimage.morphology import disk
from scipy.ndimage import convolve 
import numpy as np
from skimage.transform import resize

img=np.load('data/wood.npy');
plt.figure(figsize=[15,7])
plt.subplot(2,3,1); plt.imshow(img); plt.title('Original')
downsize =  2; plt.subplot(2,3,2); plt.imshow(resize(img,(img.shape[0] // downsize, img.shape[1] // downsize), anti_aliasing=False)); plt.title('Downsize {0}x{0}'.format(downsize))
downsize = 32; plt.subplot(2,3,3); plt.imshow(resize(img,(img.shape[0] // downsize, img.shape[1] // downsize),anti_aliasing=False)); plt.title('Downsize {0}x{0}'.format(downsize))
levels   = 16; plt.subplot(2,3,5); plt.imshow(np.floor(img*levels)); plt.title('{0} Levels'.format(levels));
levels   = 4 ; plt.subplot(2,3,6); plt.imshow(np.floor(img*levels)); plt.title('{0} Levels'.format(levels));

:::

## Sampling - Pixel size and resolution 

It is important to distinguish between pixel size and resolution

### Resolution 
The resolution is related to the optical system

- It is the effect of the optical transfer function of the acquisition system.
- Should have a greater value than the pixel size.

### Pixel size
The pixel size is

- The sample pitch between two adjacent pixels
- The smallest area represented in the image

> The sampling theorem: The pixel must be at least a factor 2 smaller than the smallest feature.

***
### Demonstrating different pixel sizes
What happens when we represent the same image with less pixels?

::: {.content-visible when-format="pdf"}
In this example we downsample the image first by a factor two. This change is barely visible when we show the image, but the number of pixels have reduced by a factor four. In the second example the image is downscaled by a factor 32 and you can clearly observe how pixelated the image is. A this level of downscaling, you can only see very coarse features in the sample.

Down-scaling is sometimes used as a method to speed up the frame rate as it radiacally reduces the number of bytes to be transfered from the detector and also the amount of data to write on disk. You should however be careful not to down scale by a too great factor as you will loose spatial information when doing so.
:::

In [ ]:
#| echo: false
#| output: true
#| fig-cap: "A sequence of images with reduced number of pixels (reductionfactor 1,2, and 32)."
#| label: fig-downsizewood
from skimage.io import imread
from skimage.morphology import disk
from scipy.ndimage import convolve 
import numpy as np
from skimage.transform import resize
import matplotlib.pyplot as plt

img=np.load('data/wood.npy');
fig,ax = plt.subplots(1,3, figsize=[15,5])
ax[0].imshow(img,cmap='gray'); plt.title('Original')

downsize =  2; 
resized  = resize(img,(img.shape[0] // downsize, img.shape[1] // downsize))
ax[1].imshow(resized, interpolation='None',cmap='gray'); ax[1].set_title('Downsize {0}x{0}'.format(downsize))

downsize = 32; 
resized  = resize(img,(img.shape[0] // downsize, img.shape[1] // downsize))
ax[2].imshow(resized,interpolation='None',cmap='gray'); ax[2].set_title('Downsize {0}x{0}'.format(downsize));

***
### Edges at different resolutions and pixel sizes

::: {.content-visible when-format="pdf"}

Finding the correct pixel size is related to the resolution of the imaging system. You can sample low resolved scenes with many pixels but then the edges will appear blurred and will essentially waste a lot data on little added value.

The example below shows what an ideal edge would look like and what it mostly looks like when we acquire our images. As you can see, the "real" edge is represented by a smooth transition spread over several pixels.

:::

![Examples of edges sampled with different pixel sizes.](figures/edges.png){#fig-pixelsize_sampling width=50%}


## Sampling - Image intensity
What happens when we reduce the number of gray-levels in the image?

::: {.content-visible when-format="pdf"}

The image intensity is determined by the response function of the imaging system. In the case of transmission imaging we are talking about the transmission of a radiation through the sample. The transmission follows Beer-Lambert's law

$$
I(x,y)=I(x,y)\cdot{}e^{-\int_L \mu(x) dx}
$$, 

this is only a simplified version. More complicated versions including the neutron energy are presented in other parts of this course.

The information captured by the detector is stored in digital form with different gray level dynamics. We are often talking about 8 or 16 bit integer when we store images. This means that each pixel can represent the measured intensity with either 256 or 65565 gray levels respectively. In the example below we demonstrate what happens when only very few gray levels are used.

:::

In [ ]:
#| echo: false
#| output: true
#| fig-cap: "Image of a bone sample as expected, as observed, and as reconstructed."
#| label: fig-levelswood
img=np.load('data/wood.npy');
fig,ax = plt.subplots(1,3,figsize=[15,7])
ax[0].imshow(img, cmap='gray'); plt.title('Original')

levels   = 16; 
lvl = np.floor(img*levels)
ax[1].imshow(lvl, cmap='gray'); ax[1].set_title('{0} Levels'.format(levels));

levels   = 4 ; 
lvl = np.floor(img*levels)
ax[2].imshow(lvl, cmap='gray'); ax[2].set_title('{0} Levels'.format(levels));

::: {.content-visible when-format="pdf"}
It is important to use as many gray levels at possible when you expose your images. The image turns patchy when you use too few levels which you can see in the example above. The patchiness reduces the precision of your evaluation, there is less margin to make estimations and decisions. The number of gray levels depend on many factors like:

- Exposure time
- Source intensity
- Conversion efficiency
- Conversion rate of the detector
- Pixel size

So, it is your task to optimize your acquisition to provide well illuminated images by changing these parameters. Some are easier than others to change and contstraint are given by the type of investigation you are doing.
:::

***
### How many bits are needed?

The number of bits you need depends on:

- Contrast difference
- Separate many different sample features
- Sensitivity to rounding errors

::: {.content-visible when-format="pdf"}

The table below gives you an idea how many bits you need to represent your image information. In the extreme you would only need a single bit per pixel (8 pixels per byte) to represent a bi-level image from a segmentation. The other extreme would be to use double precision floating point that requires 64 bits (8 bytes) per pixels. Double precision is rarely needed and single precision is mostly sufficient which saves you memory. Saving memory is in particular important when you work with 3D images.

:::

+--------------------+-------------------+-----------------------------------------+
| Few bits           | Many bits         | Floating point                          |
+====================+===================+=========================================+
| - High contrast    | - Low contrast    | - High intensity dynamics               |
| - Clean images     | - Noisy images    | - Quantification to physical properties |
| - Segmented images | - Gradual changes | - In algorithms                         |
+--------------------+-------------------+-----------------------------------------+


## The histogram

The histogram is a statistical tool to show frequency of each graylevel in the image.

::: {.content-visible when-format="pdf"}

It is essentially a plot where the you count how many times each gray level appears in the image. For a 16-bit image this would result 65565 points. This is far too detailed, therefore it is common to use bins of several gray level to reduce the level of detail in the histogram and also improve the readability. In the example below we use 100 histogram bins which look quite reasonable for this image. Chosing the number of bins depends on the image size too. Your histogram doesn't look very useful if you have too many bins compared to the available number of pixels. You can use the piece of code below to explore what happens when you change the number of bins.

:::


In [ ]:
#| echo: false
#| output: true
#| fig-cap: "Histogram of the wood sample."
#| label: fig-histogramwood
import numpy as np
import matplotlib.pyplot as plt

fig,ax = plt.subplots(1,2,figsize=(15,5))

# Compute and show a histogram
ax[0].hist(img.ravel(),bins=200)

ax[0].set_xlabel('Image value'), ax[0].set_ylabel('Number of pixels')
ax[1].imshow(img,cmap='gray');

::: {.content-visible when-format="pdf"}

It can be used for the analysis of the image as it gives you an idea which values are related to different features in the image. The histogram tells you the area covered by a give pixel value and a later section we will see how the histogram can be used to segment the images.

:::

*** 
### Details of the histogram

Let's look att different regions in the image and their representation in the histogram.

![Regions in the image connected to their position in the histogram.](figures/histogram_regions){width=50%}

***
### Histogram examples

In [ ]:
#| echo: false
#| output: true
#| fig-cap: "Histograms of different example images."
#| label: fig-histexamples
import numpy as np
import matplotlib.pyplot as plt
a=plt.imread('figures/testpattern_noisy.jpg')
b=plt.imread('figures/neutron_camera.png')
c0=plt.imread('figures/root_slices.png')
c1=plt.imread('figures/root_histogram.png')

fig,ax=plt.subplots(2,3,figsize=(10,5))
ax=ax.ravel()

ax[0].imshow(a,cmap='gray')
ax[0].set_xticks([]);ax[0].set_yticks([])
ax[0].set_title('Noisy bi-level image')
ax[3].hist(a.ravel(),bins=100)
ax[1].imshow(b,cmap='gray')
ax[1].set_title('Neutron image')
ax[1].set_xticks([]);ax[1].set_yticks([])
ax[4].hist(b.ravel(),bins=100)
ax[2].imshow(c0)
ax[2].set_title('Bivariate data')
ax[2].axis('off')
ax[5].imshow(c1)
ax[5].axis('off');

## Visualizing gray levels

The human eye is not able to resolve many intensity levels

::: {.content-visible when-format="pdf"}

In this example we demonstrate how easy or hard it can be to perceive contrast differences in an image with different number of gray levels. How well you can see the differences depends on one hand on how well your eye can resolve the contrast difference, but there are also technical issues related to how well you can see the changes. E.g. how well your screen can display the changes and even the ambient light in the room you are working in.

:::

In [ ]:
#| echo: false
#| output: true
#| fig-cap: "Demonstration of the visibility between gray levels."
#| label: fig-visibilitylevels
import numpy as np
import matplotlib.pyplot as plt
xlin = np.linspace(0,255, 256)
xx, yy = np.meshgrid(xlin, xlin)

fig, ax = plt.subplots(1,3, figsize = (15, 5))
n=32; ax[0].imshow(np.floor(xx/n), interpolation='None', cmap = 'gray');  ax[0].set_title('{0} levels'.format(256//n))
n=8;  ax[1].imshow(np.floor(xx/n), interpolation='None', cmap = 'gray');  ax[1].set_title('{0} levels'.format(256//n))
n=1;  ax[2].imshow(np.floor(xx/n), interpolation='None', cmap = 'gray');  ax[2].set_title('{0} levels'.format(256//n));

***
### Brightness
With image brightness, you can focus on narrow gray level intervals to better visually resolve local details.

![Narrow intensity intervals to highlight low (left) and high (right) graylevel regions.](figures/histogram_brightness){#fig-brightness width=60%}

::: {.content-visible when-format="pdf"}

Contrast control is often used to define which gray levels to include when you save image to file. The image is usually represented in floating point data format after some calculations and you have to limit the interval to resolve the relevant information with many gray levels and reject outliers when you convert to 8- or 16-bit integers. 

The example shows a narrow interval that mostly is useful to highlight features with small difference in contrast. The example with wider interval would is set to reject the background while most of the sample is visible. This setting may be useful for presentationations and publication where you want to boost the visibility, but is not recommended if you want to use the image in further calculations. In the latter case it is important to keep as many gray levels as possible, i.e. also include the noise flucuations in the background.

:::

***
### Contrast

Contrast controls the width of the intensity interval to use.

![Different image contrasts. Narrow interval to the left and wide interval to the right.](figures/histogram_contrast){#fig-contrast width=60%}

::: {.content-visible when-format="pdf"}

Contrast control is often used to define which gray levels to include when you save image to file. The image is usually represented in floating point data format after some calculations and you have to limit the interval to resolve the relevant information with many gray levels and reject outliers when you convert to 8- or 16-bit integers. 

The example shows a narrow interval that mostly is useful to highlight features with small difference in contrast. The example with wider interval would is set to reject the background while most of the sample is visible. This setting may be useful for presentationations and publication where you want to boost the visibility, but is not recommended if you want to use the image in further calculations. In the latter case it is important to keep as many gray levels as possible, i.e. also include the noise flucuations in the background.

:::

## 2D Intensity Images

::: {.content-visible when-format="pdf"}

Intensity images are generally represented as a 2D array of values. The following image is a 5x5 array with random numbers. Here, we create the image as a table with x,y coordinate pairs with their asociated intensity value. This is generally an unnecessary step, but we do it to show the relation between coordinates and values.

An image can be represented as a table with coordinates and the asociated value. This is however not so easy to interpret.

:::

Let's create a table

In [ ]:
#| echo: true
#| output: true
#| fig-cap: "Creating a table of image intensities and their coordinates."
#| label: tab-intensitytable
import numpy as np
import pandas as pd

basic_image = np.random.choice(range(100), size = (5,5))

xx, yy   = np.meshgrid(range(basic_image.shape[1]), range(basic_image.shape[0]))
image_df = pd.DataFrame(dict(x = xx.ravel(),
                 y = yy.ravel(),
                 Intensity = basic_image.ravel()))
image_df[['x', 'y', 'Intensity']].head(5)

***
### Viewing the data as an image

::: {.content-visible when-format="pdf"}

The table can also be shown as an image. Here, you also see the intensity values of pixel. In some cases, for very small images it can be relevant to print the pixel values, particularly when you want to follow the progress of an algorithm. But, this it not feasible for large images as the text will be so dense that it obscures the image itself.

:::

In [ ]:
#| echo: false
#| output: true
#| fig-cap: "Creating a table of image intensities and their coordinates."
#| label: fig-intensitytable

import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(1,1)
plot_image = ax1.matshow(basic_image, cmap = 'gray')
plt.colorbar(plot_image)

for _, c_row in image_df.iterrows():
    ax1.text(c_row['x'], c_row['y'], s = '%02d' % c_row['Intensity'], fontdict = dict(color = 'r'))

::: {.content-visible when-format="pdf"}

The interpretation is much easier if we instead visualize the data in as an image where the pixel values are represented by an intensity.

:::

***
### Let's colorize our image

The next step is to apply a color map (also called lookup table, LUT) to the image 

- so it is a bit more exciting 
- some features are easier to detect [@rogowitz1996]

::: {.content-visible when-format="pdf"}

It may be hard to spot subtle changes in intensity. One way is to apply a color map that translate the image intensities to a color. This color does usually not have a physical meaning but it helps you to better interpret the variations in the image.

:::



***
### Using pseudo-colormaps

- The image intensity is mostly only represented by a scalar by a gray level. 
- Which is makes it hard to see subtle changes in intensity. 

> Colormaps can help here.

In [ ]:
#| echo: false
#| output: true
#| fig-cap: "Showing our image with different colormaps."
#| label: fig-intensitytable_colors

fig, ax = plt.subplots(1,3,figsize=(15,5))
ax=ax.ravel()

a0=ax[0].matshow(basic_image, cmap = 'Blues')
ax[0].set_title('Blues')
fig.colorbar(a0,ax=ax[0],shrink=0.75)
a1=ax[1].matshow(basic_image, cmap = 'jet')
ax[1].set_title('jet')
fig.colorbar(a1,ax=ax[1],shrink=0.75)
a2=ax[2].matshow(basic_image, cmap = 'viridis')
ax[2].set_title('viridis')
fig.colorbar(a2,ax=ax[2],shrink=0.75);

## Pixelwise operations

Pixelwise operations apply scalar operations to each pixel.

- Arithmetics +, -, *, /
- Functions e.g. sin(x), exp(x), ln(x)

Statistic functions

- mean, standard deviation
- min, max

***
### Demonstrating arithmetic functions (flat field normalization)
::: {.content-visible when-format="pdf"}

Flat field normalization is a frequent operation in imaging applications. It normalizes the image of the sample by the illumination intensity, ob. Most cameras also have an intensity bias which is needed in the conversion from photons to gray levels. This is the so called dark current image which is here denoted dc. 

You can see that the original sample image has an inhomogeneous background. The back ground is flat after the normalization.

:::

$$
normed = \frac{img-dc}{ob-dc}
$$

In [ ]:
#| echo: false
#| output: true
#| fig-cap: "Demonstration of pixelwise arithmetic operations on real images."
#| label: fig-pixewise_arithmetic

img = plt.imread('data/wood_0000.tif')
ob  = plt.imread('data/ob_0000.tif')
dc  = plt.imread('data/dc_0000.tif')

normed = (img-dc)/(ob-dc)

# Visualization
fig,ax = plt.subplots(1,4,figsize=(15,4))
ax[0].imshow(img,    cmap='gray'); ax[0].set_title('Sample image')
ax[1].imshow(ob,     cmap='gray'); ax[1].set_title('Open beam image')
ax[2].imshow(dc,     cmap='gray'); ax[2].set_title('Dark current image')
ax[3].imshow(normed, cmap='gray'); ax[3].set_title('Normalized image');

## 3D images
::: {.content-visible when-format="pdf"}

For a 3D image, the position or spatial component has a 3rd dimension (z if it is a spatial, or t if it is a movie)

:::

:::: {.columns}

::: {.column width=50%}

![A cube as an example of a spatial volume in 3D.](figures/cube_10x10x10){#fig-3Dvolume width=40%}

:::

::: {.column width=50%}

![A movie as an example of a spatio-temporal volume in 3D.](figures/timeseries_visualization){#fig-3Dmovievolume width=50%}

:::

::::


***
### A 3D image as array
::: {.content-visible when-format="pdf"}
A 3D image is handled as a 3D array. It is not possible to print the 3D slice planes as a text, therefore you see the three repeated 2D arrays.
:::

In [ ]:
#| echo: true
#| output: true
#| fig-cap: "Representation of a 3D volume image as an array."
#| label: fig-3Darray

import numpy as np
vol_image = np.arange(27).reshape((3,3,3))
print(vol_image)

## 4D Images
::: {.content-visible when-format="pdf"}

There are different interpretations of what a 4D image is. The most straight forward interpretation is that it is a time series of volume images. Other cases could be that each pixel in the volume is represented by a spectrum.

:::

![](figures/4D-images){#fig-4Dimage width=50%}
